In [1]:
import xarray as xr
import pandas as pd
import glob
import os
import math

import numpy as np
import re

## Read the ppe and obs as nc files


In [2]:
ppe = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/simv4_iteration1.nc")
obs = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/obs.nc")

In [3]:

obs_dict = {"SWCF": "swcrf_toa", "LWCF": "lwcrf_toa", "TGCLDLWP": "clwp", "TMQ": "pwv",
          "CLDTOT_ISCCP": "clt_isccp", "FLUT": "olr", "PRECT": "pr","FSNTOA": "swabs_toa"}


## Process the nc ppe and obs to pd.df and series

In [4]:

lat_bins = np.arange(-85, 86, 5)  # -90 to 90 every 10 degrees
lat_labels = ((lat_bins[:-1] + lat_bins[1:]) / 2).astype(int).astype(str)  # midpoint labels
lat_labels = np.char.add("zonal_lat_",lat_labels)


In [5]:
## Take the subset by the lat range defined earlier,

ppe_zonal_list = []
obs_zonal_list = []

for cam_name, obs_name in obs_dict.items():

    ppe_da = ppe[cam_name]
    filter_tf = obs[obs_name].notnull()           ## ## Take out the na values that are in obs from the PPE 
    ppe_da_filtered = ppe_da.where(filter_tf)
    
    zonal_ppe_temp = ppe_da_filtered.mean(dim  = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_dataframe().unstack(level = 1)
    zonal_ppe_temp.columns.name = None # At this point, zonal_ppe_temp has two level in the columns
    zonal_ppe_temp.columns = ["_".join(col) for col in list(zonal_ppe_temp.columns)] # The comprehension unpack the two levels
    

    zonal_obs_temp = obs[obs_name].mean(dim = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_series()
    zonal_obs_temp.index = zonal_ppe_temp.columns

    
    ppe_zonal_list.append(zonal_ppe_temp)
    obs_zonal_list.append(zonal_obs_temp)

ppe_zonal_pd = pd.concat(ppe_zonal_list, axis = 1)
obs_zonal_pd = pd.concat(obs_zonal_list)


In [6]:
### Added locations for specific climatologies
man_sel_locations1 = pd.Series({"cli": "SWCF", "lat_min": -6,"lat_max": -4, "lon_min":  141, "lon_max": 144})
man_sel_locations2 = pd.Series({"cli": "SWCF", "lat_min": 7,"lat_max": 9, "lon_min":  235, "lon_max": 240})



manul_ppe_info = pd.concat([man_sel_locations1, man_sel_locations2], axis  = 1).transpose()
manul_ppe_info

,cli,lat_min,lat_max,lon_min,lon_max
0,SWCF,-6,-4,141,144
1,SWCF,7,9,235,240


In [7]:
## Run interations to take the subset based on the cell above
ppe_manual_list = []
obs_manual_list = []
manual_name_list = []

for row_ind, row in manul_ppe_info.iterrows():
    
    temp_obs = obs[obs_dict[row.cli]].sel(lat = slice(row.lat_min, row.lat_max), lon = slice(row.lon_min, row.lon_max)).mean(dim = ["lat", "lon"]).values
    if ~np.isnan(temp_obs):
        temp_ppe = ppe[row.cli].sel(lat = slice(row.lat_min, row.lat_max), lon = slice(row.lon_min, row.lon_max)).mean(dim = ["lat", "lon"]).to_dataframe()
        
        manual_name_list.append("_".join(row.astype(str)))
        ppe_manual_list.append(temp_ppe)
        obs_manual_list.append(temp_obs)
    else:
        print("??")


In [8]:
ppe_manual_list = pd.concat(ppe_manual_list,axis = 1)
obs_manual_list = pd.Series(obs_manual_list)

ppe_manual_list.columns = manual_name_list
obs_manual_list.index = manual_name_list

In [9]:
ppe_manual_list

,SWCF_-6_-4_141_144,SWCF_7_9_235_240
ppe_ind,,
1,-128.715744,-87.976289
3,-104.409674,-89.894234
4,-125.420282,-102.611100
5,-131.558543,-77.443123
6,-103.527699,-53.324356
...,...,...
86,-121.412717,-101.191075
87,-105.989846,-73.850319
88,-130.176916,-104.578004


In [12]:
ppe_pd = pd.concat([ppe_zonal_pd, ppe_manual_list], axis = 1)
obs_pd = pd.concat([obs_zonal_pd, obs_manual_list])


In [13]:
ppe_pd.head()

,SWCF_zonal_lat_-82,SWCF_zonal_lat_-77,SWCF_zonal_lat_-72,SWCF_zonal_lat_-67,SWCF_zonal_lat_-62,SWCF_zonal_lat_-57,SWCF_zonal_lat_-52,SWCF_zonal_lat_-47,SWCF_zonal_lat_-42,SWCF_zonal_lat_-37,...,FSNTOA_zonal_lat_47,FSNTOA_zonal_lat_52,FSNTOA_zonal_lat_57,FSNTOA_zonal_lat_62,FSNTOA_zonal_lat_67,FSNTOA_zonal_lat_72,FSNTOA_zonal_lat_77,FSNTOA_zonal_lat_82,SWCF_-6_-4_141_144,SWCF_7_9_235_240
ppe_ind,,,,,,,,,,,,,,,,,,,,,
1,-1.775323,-3.677040,-8.188253,-26.007094,-55.478150,-73.107625,-77.024634,-73.212884,-64.607686,-54.185431,...,191.494925,167.835081,146.375524,122.742515,102.767963,87.831024,76.362725,71.129395,-128.715744,-87.976289
3,-1.618221,-3.374800,-8.302124,-26.239062,-55.215606,-72.102811,-75.852205,-72.355755,-63.536108,-53.471915,...,194.268145,172.547603,151.700622,128.713602,106.657581,88.535728,76.195900,70.599075,-104.409674,-89.894234
4,-1.765804,-3.844527,-8.594140,-27.004102,-56.849576,-74.685015,-78.965654,-77.357821,-74.099307,-67.848767,...,190.954915,167.326394,146.043029,123.389661,103.185150,86.342926,74.753110,69.292476,-125.420282,-102.611100
5,-2.353393,-4.225666,-8.897521,-27.351285,-56.519404,-73.319435,-76.810730,-74.351819,-69.418165,-63.010395,...,189.980311,167.055944,144.121663,121.062649,100.773901,85.780473,74.181953,68.465038,-131.558543,-77.443123
6,-1.377978,-3.060643,-7.356543,-24.552219,-52.189145,-68.319704,-71.086715,-68.272482,-62.101780,-52.859389,...,192.232098,169.226385,148.078477,125.528020,104.687790,88.765484,76.183082,70.918957,-103.527699,-53.324356


In [14]:
#ppe_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/ppe_zonal_v4_iteration1.csv", index = True)
#obs_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/obs_zonal_v4_iteration1.csv", index = True)
